# 🤖 Stock Recommender — Step 4: FinBERT Fine-Tuning + Feature Extraction

**Architecture (Path C — most powerful):**

```
Step 4A  Split 70/15/15 stratified          ← prevents leakage
Step 4B  Frozen FinBERT → sentiment probs   ← 7 features
Step 4C  Fine-tune FinBERT on TRAIN only    ← new BUY/HOLD/SELL head
Step 4D  Fine-tuned probs on ALL data       ← 3 features
Step 4E  [CLS] embeddings → PCA 32 dims     ← 32 features
Step 4F  low_relevance + divergence_score   ← 2 features
Step 4G  Merge, validate, save              ← 73 total features
```

**Why split before fine-tuning:**
Fine-tuning on all data then using predictions as features in Step 5
would leak label information. The fix: fine-tune only on the 70% train
split, then extract predictions on all splits using the saved checkpoint.

**Total features entering Step 5:**

| Source | Features |
|---|---|
| Step 3 NLP/lexicon | 29 |
| Frozen FinBERT (sentiment) | 7 |
| Fine-tuned FinBERT (price probs) | 3 |
| CLS embeddings PCA | 32 |
| Engineered (low_rel, divergence) | 2 |
| **Total** | **73** |


## ⚙️ 0. Install & Imports

In [ ]:
import subprocess, sys
pkgs = ["transformers==4.40.0","torch","pandas","numpy",
        "matplotlib","seaborn","scipy","scikit-learn","tqdm"]
for p in pkgs:
    subprocess.run([sys.executable,"-m","pip","install",p,"-q"], check=False)

import re, warnings, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax
from scipy.stats import spearmanr
from scipy.stats import entropy as sh_entropy
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 110)
plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

LABEL_MAP = {0: "SELL", 1: "HOLD", 2: "BUY"}
LABEL_PAL = {0: "#F44336", 1: "#9E9E9E", 2: "#4CAF50"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU — Go to Runtime > Change Runtime Type > T4 GPU")
    print("Fine-tuning on CPU will take 3-4 hours. GPU strongly recommended.")


## 📂 1. Load Step 3 Output

In [ ]:
df = pd.read_csv("step3_nlp.csv")
df['pub_dt']   = pd.to_datetime(df['published_at'], utc=True, errors='coerce')
df['pub_date'] = pd.to_datetime(df['pub_date'])
df['title_clean'] = df['title_clean'].fillna(df['title'])

print(f"Loaded  : {len(df):,} rows, {df.shape[1]} columns")
print(f"Tickers : {df['ticker'].nunique()}")
print(f"Labels  : {df['label'].value_counts().sort_index().to_dict()}")


## ✂️ 2. Train / Val / Test Split (70 / 15 / 15)

**This split happens BEFORE fine-tuning to prevent leakage.**
FinBERT will only see training labels during fine-tuning.
Predictions on val/test come from the saved checkpoint — not from
a model that saw those labels.


In [ ]:
RANDOM_SEED = 42
TEST_SIZE   = 0.15
VAL_SIZE    = 0.15  # of total (not of remaining)

# Stratify on label to preserve class distribution across splits
idx_all = df.index.tolist()

idx_trainval, idx_test = train_test_split(
    idx_all,
    test_size=TEST_SIZE,
    stratify=df.loc[idx_all, 'label'],
    random_state=RANDOM_SEED
)

# val_frac of trainval = VAL_SIZE of total
val_frac = VAL_SIZE / (1 - TEST_SIZE)
idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=val_frac,
    stratify=df.loc[idx_trainval, 'label'],
    random_state=RANDOM_SEED
)

df['split'] = 'train'
df.loc[idx_val,  'split'] = 'val'
df.loc[idx_test, 'split'] = 'test'

print("SPLIT SIZES:")
for split in ['train','val','test']:
    sub = df[df['split']==split]
    n   = len(sub)
    lv  = sub['label'].value_counts(normalize=True).sort_index()
    print(f"  {split:5s}: {n:,}  ({n/len(df)*100:.1f}%)  "
          f"SELL={lv.get(0,0)*100:.1f}%  HOLD={lv.get(1,0)*100:.1f}%  "
          f"BUY={lv.get(2,0)*100:.1f}%")

# Save split indices for Step 5 to use the SAME split
split_df = df[['split']].copy()
split_df.to_csv("step4_split_indices.csv")
print()
print("Saved: step4_split_indices.csv")


## 🧊 3. Frozen FinBERT — Sentiment Feature Extraction

In [ ]:
MODEL_NAME = "ProsusAI/finbert"
print(f"Loading {MODEL_NAME} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
frozen_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
frozen_model = frozen_model.to(DEVICE)
frozen_model.eval()

id2label = frozen_model.config.id2label
print(f"Label map: {id2label}")

POS_IDX = [k for k,v in id2label.items() if v.lower()=='positive'][0]
NEG_IDX = [k for k,v in id2label.items() if v.lower()=='negative'][0]
NEU_IDX = [k for k,v in id2label.items() if v.lower()=='neutral'][0]

# Sanity check
test_sents = [
    "Nvidia earnings crush estimates, revenue up 122 percent",
    "Tesla misses deliveries, shares plunge 8 percent",
    "Apple reports quarterly results in line with expectations",
]
print()
print("FROZEN FINBERT SANITY CHECK:")
for s in test_sents:
    enc = tokenizer(s, return_tensors='pt',
                    truncation=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        probs = softmax(frozen_model(**enc).logits.cpu().numpy()[0])
    pred = id2label[int(probs.argmax())]
    print(f"  [{pred:8s} {probs.max():.2f}]  {s}")


In [ ]:
def frozen_inference(texts, batch_size=32):
    pos_l, neg_l, neu_l = [], [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="Frozen FinBERT"):
        batch = texts[i: i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=128, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            probs = softmax(frozen_model(**enc).logits.cpu().numpy(), axis=1)
        pos_l.extend(probs[:, POS_IDX].tolist())
        neg_l.extend(probs[:, NEG_IDX].tolist())
        neu_l.extend(probs[:, NEU_IDX].tolist())

    fb = pd.DataFrame({'fb_pos': pos_l, 'fb_neg': neg_l, 'fb_neu': neu_l})
    fb['fb_net']     = fb['fb_pos'] - fb['fb_neg']
    fb['fb_conf']    = fb[['fb_pos','fb_neg','fb_neu']].max(axis=1)
    fb['fb_entropy'] = fb[['fb_pos','fb_neg','fb_neu']].apply(
        lambda r: float(sh_entropy(r.values + 1e-9)), axis=1)
    fb['fb_label']   = fb[['fb_pos','fb_neg','fb_neu']].idxmax(axis=1).str.replace('fb_','')
    return fb

t0 = time.time()
fb_df = frozen_inference(df['title_clean'].tolist())
elapsed_frozen = round(time.time() - t0)
print(f"Frozen inference done in {elapsed_frozen}s")

# Rolling 3-article momentum per ticker (shift=1 to avoid leakage)
df_s = df[['ticker','pub_dt']].copy()
df_s['fb_net_tmp'] = fb_df['fb_net'].values
df_s = df_s.sort_values(['ticker','pub_dt'])
df_s['fb_momentum'] = (
    df_s.groupby('ticker')['fb_net_tmp']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
df['fb_momentum'] = df_s['fb_momentum'].reindex(df.index).fillna(0.0)
for col in ['fb_pos','fb_neg','fb_neu','fb_net','fb_conf','fb_entropy','fb_label']:
    df[col] = fb_df[col].values
print("Frozen FinBERT features attached. ✅")


## 🔥 4. Fine-Tune FinBERT on Price-Reaction Labels

Fine-tuning replaces FinBERT's original 3-class head (positive/negative/neutral)
with a new head trained directly on our BUY/HOLD/SELL price labels.

The model learns that:
- "JNJ tops earnings" can be HOLD (priced in before publication)
- "Airtel plans $1B fundraise" can be SELL (dilution concern)
- Specific linguistic patterns that precede counter-intuitive price moves

**Only the TRAIN split is used here.** Val split is used for early stopping.


In [ ]:
class HeadlineDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            list(texts), padding=True, truncation=True,
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


# ── Dataset objects ────────────────────────────────────────────────────────────
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)

train_dataset = HeadlineDataset(
    train_df['title_clean'], train_df['label'], tokenizer)
val_dataset   = HeadlineDataset(
    val_df['title_clean'],   val_df['label'],   tokenizer)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")


In [ ]:
# ── Class weights to handle 1.53x imbalance ───────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

cw = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_df['label'].values
)
class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
print(f"Class weights: SELL={cw[0]:.3f}  HOLD={cw[1]:.3f}  BUY={cw[2]:.3f}")

# ── Fine-tuning model — fresh classification head ──────────────────────────────
ft_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True   # replaces original 3-class head with new one
)
ft_model = ft_model.to(DEVICE)

# ── Optimiser with layer-wise learning rates ───────────────────────────────────
# Lower LR for pretrained layers, higher for new head
no_decay   = ['bias', 'LayerNorm.weight']
head_params = ['classifier']

param_groups = [
    {'params': [p for n,p in ft_model.named_parameters()
                if not any(nd in n for nd in no_decay)
                and not any(h in n for h in head_params)],
     'lr': 2e-5, 'weight_decay': 0.01},

    {'params': [p for n,p in ft_model.named_parameters()
                if any(nd in n for nd in no_decay)
                and not any(h in n for h in head_params)],
     'lr': 2e-5, 'weight_decay': 0.0},

    {'params': [p for n,p in ft_model.named_parameters()
                if any(h in n for h in head_params)],
     'lr': 1e-4, 'weight_decay': 0.01},
]

EPOCHS      = 4
WARMUP_FRAC = 0.1
total_steps = len(train_loader) * EPOCHS

optimizer  = AdamW(param_groups)
scheduler  = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_FRAC),
    num_training_steps=total_steps
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"Epochs          : {EPOCHS}")
print(f"Total steps     : {total_steps}")
print(f"Warmup steps    : {int(total_steps * WARMUP_FRAC)}")
print(f"Estimated time  : ~{total_steps // 60 + 1} min on T4")


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def run_epoch(model, loader, optimizer=None, scheduler=None, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1


# ── Training loop ──────────────────────────────────────────────────────────────
history = {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[]}
best_val_f1   = -1
best_epoch    = 0
CKPT_PATH     = "step4_finbert_best.pt"

print("TRAINING:")
print(f"{'Epoch':>6} {'Tr Loss':>9} {'Tr F1':>8} {'Val Loss':>10} {'Val F1':>8} {'Best':>6}")
print("-" * 55)

t_start = time.time()
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_f1 = run_epoch(
        ft_model, train_loader, optimizer, scheduler, train=True)
    vl_loss, vl_acc, vl_f1 = run_epoch(
        ft_model, val_loader, train=False)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_f1'].append(tr_f1)
    history['val_f1'].append(vl_f1)

    is_best = vl_f1 > best_val_f1
    if is_best:
        best_val_f1 = vl_f1
        best_epoch  = epoch
        torch.save(ft_model.state_dict(), CKPT_PATH)

    marker = " ← best" if is_best else ""
    print(f"  {epoch:3d}  {tr_loss:9.4f}  {tr_f1:8.4f}  "
          f"{vl_loss:10.4f}  {vl_f1:8.4f}{marker}")

elapsed_ft = round(time.time() - t_start)
print()
print(f"Training done in {elapsed_ft}s")
print(f"Best checkpoint : epoch {best_epoch}  val_f1={best_val_f1:.4f}")
print(f"Saved: {CKPT_PATH}")


In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_x = list(range(1, EPOCHS + 1))

axes[0].plot(epochs_x, history['train_loss'], 'o-', color='#2196F3', label='Train')
axes[0].plot(epochs_x, history['val_loss'],   's-', color='#F44336', label='Val')
axes[0].axvline(best_epoch, color='green', linestyle='--', linewidth=1.5,
                label=f'Best epoch {best_epoch}')
axes[0].set_title("Loss Curves", fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("CrossEntropy Loss")
axes[0].legend()

axes[1].plot(epochs_x, history['train_f1'], 'o-', color='#2196F3', label='Train')
axes[1].plot(epochs_x, history['val_f1'],   's-', color='#F44336', label='Val')
axes[1].axvline(best_epoch, color='green', linestyle='--', linewidth=1.5,
                label=f'Best epoch {best_epoch}')
axes[1].set_title("Macro F1 Curves", fontweight='bold')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].legend()

plt.suptitle("FinBERT Fine-Tuning Progress", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("step4_training_curves.png", bbox_inches='tight', dpi=150)
plt.show()

print(f"Final val Macro F1 (best checkpoint): {best_val_f1:.4f}")
print("Expected range: 0.42-0.55 at epoch 4")
print("(Higher is better; above 0.45 = FinBERT is learning price-reaction patterns)")


## 🎯 5. Extract Fine-Tuned Probabilities + CLS Embeddings

Load the best checkpoint and run inference on ALL splits.
Outputs:
- `ft_buy`, `ft_hold`, `ft_sell` — fine-tuned probability for each class
- `emb_*` — raw [CLS] embeddings (768-dim) before PCA compression


In [ ]:
# Load best checkpoint
ft_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
ft_model.eval()
print(f"Loaded best checkpoint (epoch {best_epoch})")

# Label mapping for fine-tuned model output
# Our fine-tuned model: label 0=SELL, 1=HOLD, 2=BUY
FT_SELL_IDX = 0
FT_HOLD_IDX = 1
FT_BUY_IDX  = 2


def extract_probs_and_embeddings(texts, batch_size=32):
    sell_l, hold_l, buy_l = [], [], []
    cls_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Fine-tuned FinBERT"):
        batch = texts[i: i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=128, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            outputs = ft_model(**enc, output_hidden_states=True)

        # Classification probabilities
        probs = softmax(outputs.logits.cpu().numpy(), axis=1)
        sell_l.extend(probs[:, FT_SELL_IDX].tolist())
        hold_l.extend(probs[:, FT_HOLD_IDX].tolist())
        buy_l.extend( probs[:, FT_BUY_IDX].tolist())

        # [CLS] token from last hidden layer — shape (batch, 768)
        last_hidden = outputs.hidden_states[-1]
        cls = last_hidden[:, 0, :].cpu().numpy()  # [CLS] is index 0
        cls_embeddings.append(cls)

    probs_df = pd.DataFrame({
        'ft_sell': sell_l,
        'ft_hold': hold_l,
        'ft_buy' : buy_l,
    })
    probs_df['ft_net'] = probs_df['ft_buy'] - probs_df['ft_sell']
    probs_df['ft_conf'] = probs_df[['ft_sell','ft_hold','ft_buy']].max(axis=1)

    all_cls = np.vstack(cls_embeddings)
    return probs_df, all_cls


texts = df['title_clean'].tolist()
t0 = time.time()
ft_probs_df, cls_matrix = extract_probs_and_embeddings(texts)
elapsed_ft_inf = round(time.time() - t0)
print(f"Inference done in {elapsed_ft_inf}s")
print(f"CLS matrix shape: {cls_matrix.shape}")
print()
print(ft_probs_df.head(5).round(4).to_string())


## 📐 6. PCA Compression of CLS Embeddings (768 → 32 dims)

In [ ]:
# PCA fit ONLY on training data — transform applied to all
# This prevents test/val statistics from influencing the PCA axes

train_mask = (df['split'] == 'train').values
train_cls  = cls_matrix[train_mask]

scaler = StandardScaler()
train_cls_scaled = scaler.fit_transform(train_cls)
all_cls_scaled   = scaler.transform(cls_matrix)

N_COMPONENTS = 32
pca = PCA(n_components=N_COMPONENTS, random_state=42)
pca.fit(train_cls_scaled)

pca_all = pca.transform(all_cls_scaled)
explained_var = pca.explained_variance_ratio_.cumsum()

print(f"PCA components       : {N_COMPONENTS}")
print(f"Variance explained   : {explained_var[-1]*100:.1f}%")
print(f"First 10 components  : {explained_var[9]*100:.1f}%")
print()

# Variance explained plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, N_COMPONENTS + 1),
        pca.explained_variance_ratio_.cumsum() * 100,
        'o-', color='#673AB7', linewidth=2, markersize=5)
ax.axhline(explained_var[-1]*100, color='red', linestyle='--',
           label=f'{explained_var[-1]*100:.1f}% at {N_COMPONENTS} components')
ax.set_xlabel("Number of PCA Components")
ax.set_ylabel("Cumulative Variance Explained (%)")
ax.set_title("PCA Variance Explained — FinBERT CLS Embeddings", fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("step4_pca_variance.png", bbox_inches='tight', dpi=150)
plt.show()

# Attach PCA features to df
pca_cols = [f'pca_{i}' for i in range(N_COMPONENTS)]
pca_df   = pd.DataFrame(pca_all, columns=pca_cols, index=df.index)
df = pd.concat([df, pca_df], axis=1)

# Save PCA + scaler for inference
with open("step4_pca_scaler.pkl", "wb") as f:
    pickle.dump({'pca': pca, 'scaler': scaler}, f)
print("Saved: step4_pca_scaler.pkl  (needed for Step 5 inference)")


## 🔍 7. Low-Relevance Flag + Divergence Score

In [ ]:
RELEVANCE_KW = {
    'profit','loss','revenue','sales','earnings','results','quarterly',
    'q1','q2','q3','q4','eps','pat','ebitda','margin','guidance',
    'upgrade','downgrade','target','rating','outperform','underperform',
    'merger','acquisition','ipo','dividend','buyback','split','deal',
    'surge','plunge','jump','fall','drop','rise','crash','rally','decline',
    'debt','default','bankruptcy','npa','lawsuit','fine','fraud','probe',
    'percent','crore','billion','million','lakh',
    'nifty','sensex','circuit','fii','dii','sebi','rbi',
}

def is_low_rel(title):
    t = str(title).lower()
    return 0 if any(kw in t for kw in RELEVANCE_KW) else 1

df['low_relevance'] = df['title_clean'].apply(is_low_rel)

# Divergence: sign(fb_net) * sign(ret_1d)
# +1 = sentiment agrees with price direction
# -1 = counter-intuitive reaction (training signal only)
df['divergence_score'] = (
    np.sign(fb_df['fb_net'].values) * np.sign(df['ret_1d'].values)
).astype(float)

n_low = df['low_relevance'].sum()
agree_pct    = (df['divergence_score'] ==  1).mean() * 100
disagree_pct = (df['divergence_score'] == -1).mean() * 100

print(f"Low-relevance articles : {n_low:,} ({n_low/len(df)*100:.1f}%)")
print(f"Sentiment AGREE        : {agree_pct:.1f}%")
print(f"Sentiment DISAGREE     : {disagree_pct:.1f}%")
print()
print("Samples flagged as low-relevance:")
for t in df[df['low_relevance']==1]['title_clean'].sample(min(6, n_low), random_state=42):
    print(f"  - {t[:85]}")


## 🔗 8. Merge All Features

In [ ]:
for col in ['ft_sell','ft_hold','ft_buy','ft_net','ft_conf']:
    df[col] = ft_probs_df[col].values

STEP3_FEATURES = [
    'kw_strong_bull','kw_weak_bull','kw_strong_bear','kw_weak_bear',
    'kw_neutral','kw_event','kw_analyst','kw_india_bull','kw_india_bear',
    'kw_net','has_price_target','pct_magnitude',
    'token_count','flag_extreme_move','extreme_move_pct',
    'pub_hour','pub_dow','pub_month','pub_quarter',
    'is_us_premarket','is_us_market','is_in_premarket','is_in_market',
    'is_weekend','is_friday_close',
    'is_high_credibility','is_india','daily_news_vol','log_news_vol',
]

FROZEN_FB_FEATURES = [
    'fb_pos','fb_neg','fb_neu','fb_net','fb_conf','fb_entropy','fb_momentum',
]

FT_FEATURES = ['ft_sell','ft_hold','ft_buy','ft_net','ft_conf']

ENGINEERED_FEATURES = ['low_relevance','divergence_score']

PCA_FEATURES = [f'pca_{i}' for i in range(32)]

ALL_FEATURES = (STEP3_FEATURES + FROZEN_FB_FEATURES +
                FT_FEATURES + ENGINEERED_FEATURES + PCA_FEATURES)

print(f"Step 3 features      : {len(STEP3_FEATURES)}")
print(f"Frozen FinBERT       : {len(FROZEN_FB_FEATURES)}")
print(f"Fine-tuned FinBERT   : {len(FT_FEATURES)}")
print(f"Engineered           : {len(ENGINEERED_FEATURES)}")
print(f"PCA embeddings       : {len(PCA_FEATURES)}")
print(f"Total                : {len(ALL_FEATURES)}")
print()

# Null check
issues = 0
for col in ALL_FEATURES:
    if col not in df.columns:
        print(f"  MISSING: {col}")
        issues += 1
    elif df[col].isna().sum() > 0:
        print(f"  {df[col].isna().sum()} NULLS: {col}")
        issues += 1
if issues == 0:
    print(f"All {len(ALL_FEATURES)} features present and complete. ✅")


## 📊 9. Feature Importance Ranking (Spearman vs Label)

In [ ]:
correlations = {}
for col in ALL_FEATURES:
    try:
        r, p = spearmanr(df[col].fillna(0), df['label'])
        correlations[col] = (r, abs(r), p)
    except Exception:
        correlations[col] = (0, 0, 1)

corr_df = pd.DataFrame(
    [(f, r, ar, p) for f,(r,ar,p) in correlations.items()],
    columns=['feature','spearman_r','abs_r','p_value']
).sort_values('abs_r', ascending=False).reset_index(drop=True)

print("ALL FEATURES RANKED BY |SPEARMAN r| WITH PRICE LABEL:")
print("="*68)
print(f"  {'#':<4} {'Feature':<30} {'r':>8} {'|r|':>8}  sig")
print("="*68)
for i, row in corr_df.iterrows():
    sig  = "***" if row['p_value']<0.001 else "** " if row['p_value']<0.01 else "*  " if row['p_value']<0.05 else "   "
    src  = "FT" if row['feature'].startswith('ft_') else "FB" if row['feature'].startswith('fb_') else "PC" if row['feature'].startswith('pca') else "NL"
    bar  = chr(9608) * int(row['abs_r'] * 30)
    print(f"  {i+1:<4} [{src}] {row['feature']:<28} {row['spearman_r']:>+8.4f} {row['abs_r']:>8.4f} {sig}  {bar}")

print()
print("Key: FT=fine-tuned  FB=frozen  PC=PCA  NL=NLP/lexicon")
print("*** p<0.001  ** p<0.01  * p<0.05")


In [ ]:
# Feature importance chart — top 30 only (PCA dims are many)
top30 = corr_df.head(30)
color_map = {
    'FT': '#E91E63',  # pink — fine-tuned
    'FB': '#673AB7',  # purple — frozen
    'PC': '#FF9800',  # orange — PCA
    'NL': '#2196F3',  # blue — NLP/lexicon
}
def get_color(feat):
    if feat.startswith('ft_'):  return color_map['FT']
    if feat.startswith('fb_'):  return color_map['FB']
    if feat.startswith('pca'):  return color_map['PC']
    return color_map['NL']

colors = [get_color(f) for f in top30['feature']]
fig, ax = plt.subplots(figsize=(14, 10))
ax.barh(top30['feature'][::-1], top30['abs_r'][::-1],
        color=colors[::-1], alpha=0.85, edgecolor='white')
ax.set_xlabel("|Spearman r| with Price Label", fontsize=12)
ax.set_title("Top 30 Feature Predictive Power", fontweight='bold', fontsize=13)
ax.axvline(0.05, color='gray', linestyle='--', linewidth=1, label='0.05')
ax.axvline(0.10, color='red',  linestyle='--', linewidth=1, label='0.10')

from matplotlib.patches import Patch
legend_els = [
    Patch(color='#E91E63', label='Fine-tuned FinBERT'),
    Patch(color='#673AB7', label='Frozen FinBERT'),
    Patch(color='#FF9800', label='PCA embeddings'),
    Patch(color='#2196F3', label='NLP/Lexicon'),
]
ax.legend(handles=legend_els, loc='lower right')
plt.tight_layout()
plt.savefig("step4_feature_importance.png", bbox_inches='tight', dpi=150)
plt.show()


## ✅ 10. Fine-Tuned Model Validation

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)

# Evaluate on TEST split using fine-tuned probabilities
test_mask = (df['split'] == 'test').values
y_true    = df.loc[test_mask, 'label'].values
y_pred    = ft_probs_df.loc[test_mask, ['ft_sell','ft_hold','ft_buy']].values.argmax(axis=1)

print("FINE-TUNED FINBERT — TEST SET PERFORMANCE:")
print("="*60)
print(classification_report(
    y_true, y_pred,
    target_names=['SELL','HOLD','BUY'],
    digits=4
))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['SELL','HOLD','BUY']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title("Fine-Tuned FinBERT — Test Set Confusion Matrix", fontweight='bold')
plt.tight_layout()
plt.savefig("step4_ft_confusion.png", bbox_inches='tight', dpi=150)
plt.show()

print()
print("NOTE: These are FinBERT-standalone results.")
print("Step 5 ensemble (FinBERT + XGBoost + LightGBM) will be significantly higher.")


## 💾 11. Save All Outputs

In [ ]:
SAVE_COLS = (
    ['id','ticker','region','title','title_clean',
     'published_at','pub_date','pub_hour','pub_dow',
     'label','label_1d','label_2d','label_3d',
     'horizons_agree','ret_1d','ret_2d','ret_3d',
     'source_credibility','split','fb_label']
    + ALL_FEATURES
)
SAVE_COLS = list(dict.fromkeys([c for c in SAVE_COLS if c in df.columns]))
df[SAVE_COLS].to_csv("step4_features.csv", index=False)

counts = df['label'].value_counts().sort_index()
total  = len(df)
imb    = counts.max() / counts.min()
top1   = corr_df.iloc[0]['feature']
top2   = corr_df.iloc[1]['feature']

# Fine-tuned standalone F1 (test set)
test_mask = (df['split'] == 'test').values
y_true = df.loc[test_mask, 'label'].values
y_pred = ft_probs_df.loc[test_mask, ['ft_sell','ft_hold','ft_buy']].values.argmax(axis=1)
ft_f1  = f1_score(y_true, y_pred, average='macro')

print("=" * 70)
print("STEP 4 COMPLETE — FINBERT FINE-TUNED + ALL FEATURES EXTRACTED")
print("=" * 70)
print(f"  Articles              : {total:,}")
print(f"  Fine-tune epochs      : {EPOCHS}  (best: epoch {best_epoch})")
print(f"  Best val Macro F1     : {best_val_f1:.4f}")
print(f"  FinBERT test Macro F1 : {ft_f1:.4f}  (standalone, before ensemble)")
print()
print(f"  Feature breakdown:")
print(f"    Step 3 NLP/lexicon  : {len(STEP3_FEATURES)}")
print(f"    Frozen FinBERT      : {len(FROZEN_FB_FEATURES)}")
print(f"    Fine-tuned FinBERT  : {len(FT_FEATURES)}")
print(f"    PCA embeddings      : {len(PCA_FEATURES)}")
print(f"    Engineered          : {len(ENGINEERED_FEATURES)}")
print(f"    TOTAL               : {len(ALL_FEATURES)}")
print()
for lv in [0,1,2]:
    cnt = counts[lv]
    print(f"  {LABEL_MAP[lv]:4s}: {cnt:,}  ({cnt/total*100:.1f}%)")
print(f"  Imbalance  : {imb:.2f}x")
print()
print(f"  Top Spearman feature  : {top1}")
print(f"  2nd Spearman feature  : {top2}")
print()
print("  Saved files:")
print("    step4_features.csv     — 73 features + labels + split")
print("    step4_split_indices.csv — train/val/test split")
print("    step4_finbert_best.pt   — fine-tuned checkpoint")
print("    step4_pca_scaler.pkl    — PCA + StandardScaler for inference")
print()
print("  STEP 5 PLAN:")
print("    XGBoost + LightGBM on all 73 features (class_weight=balanced)")
print("    Soft ensemble: 0.4 * FinBERT + 0.3 * XGBoost + 0.3 * LightGBM")
print("    Isotonic calibration on ensemble probabilities")
print("    SHAP attribution on top 20 features")
print("    Targets: Accuracy >62%  Macro F1 >0.60")
print()
print("Share the Spearman table + FinBERT test F1 before Step 5.")
